In [ ]:
# Install from github if running on Colab
!pip install git+https://github.com/maxiQE/fellowship_sim.git

# Rime — interactive sim

Run cells top-to-bottom once to initialise, then iterate on **Ability casts** + **Damage report**.

In [1]:
import random

from fellowship_sim import configure_logging
from fellowship_sim.base_classes import Enemy, Gem, HeroicTrait, Legendary, MasterTrait, State, Weapon
from fellowship_sim.base_classes.stats import RawStatsFromScores
from fellowship_sim.rime import Talent
from fellowship_sim.rime.entity import Rime
from fellowship_sim.rime.setup import RimeSetup

## General setup

In [2]:
# TRACE / DEBUG : verbose ability resolution
# INFO          : damage events
# SUCCESS       : important effects
# WARNING       : problems only
LOG_LEVEL = "INFO"
NUM_TARGETS = 5
SEED = 1234

configure_logging(LOG_LEVEL)

## Character setup

In [3]:
character_setup = RimeSetup(
    initial_spirit_points=130,
    initial_winter_orbs=5,
    raw_stats=RawStatsFromScores(
        main_stat=2444.0,
        crit_score=1198,
        expertise_score=1572,
        haste_score=1239,
        spirit_score=500,
    ),
    legendary=Legendary.NECK,
    weapon_ability=Weapon.CHRONOSHIFT,
    master_trait=MasterTrait.VISIONS_OF_GRANDEUR,
    heroic_traits=[
        HeroicTrait.WILLFUL_MOMENTUM,
        HeroicTrait.KINDLING,
    ],
    talents=[
        Talent.WINTERS_EMBRACE,
        Talent.BURSTBOLTER,
        Talent.ICY_FLOW,
        Talent.AVALANCHE,
        Talent.GREATER_GLACIAL_BLAST,
        Talent.FROSTWEAVERS_WRATH,
        Talent.BITING_COLD,
        Talent.WISDOM_OF_THE_NORTH,
    ],
    gem_power={
        Gem.BLUE: 2664,
        Gem.PURPLE: 1620,
        Gem.RED: 162,
    },
    sets=["Drakheim's Absolution"],
)

## Scenario

Call `reset_sim()` at the top of any ability sequence to start from a clean slate.

In [4]:
state: State
enemies: list[Enemy]
target: Enemy
rime: Rime

state = State(rng=random.Random(x=SEED))
enemies = [Enemy(state=state) for _ in range(NUM_TARGETS)]
target = enemies[0]
rime = character_setup.finalize(state)

def reset_sim() -> None:
    global state, enemies, target, rime
    state = State(rng=random.Random(x=SEED))
    enemies = [Enemy(state=state) for _ in range(NUM_TARGETS)]
    target = enemies[0]
    rime = character_setup.finalize(state)

reset_sim()

## Ability casts

In [5]:
reset_sim()

rime.wrath_of_winter.cast(target)
rime.chronoshift.cast(target)
rime.wait(40)

print(f"{rime.chronoshift.cooldown = :.1f}s  charges={rime.chronoshift.charges}")

SUCCESS  |    0.00 | Starting cast: Wrath Of Winter
INFO     |    1.25 | Rime orb gain: orbs=5
SUCCESS  |    1.25 | Starting cast: Chronoshift
INFO     |    1.35 |   3775 dmg by Flight Of The Navir on Enemy(13, dmg taken=3775)
INFO     |    1.35 |   3775 dmg by Flight Of The Navir on Enemy(17, dmg taken=3775)
INFO     |    1.35 |   8554 dmg by Flight Of The Navir on Enemy(17, dmg taken=12328)
INFO     |    2.35 |  72560 dmg by Chronoshift on Enemy(13, dmg taken=76335)
INFO     |    2.35 |  72560 dmg by Chronoshift on Enemy(14, dmg taken=72560)
INFO     |    2.35 |  32021 dmg by Chronoshift on Enemy(15, dmg taken=32021)
INFO     |    2.35 |  32021 dmg by Chronoshift on Enemy(16, dmg taken=32021)
INFO     |    2.35 |  72560 dmg by Chronoshift on Enemy(17, dmg taken=84889)
INFO     |    3.35 | 140882 dmg by kindling_dot(7.0s) on Enemy(13, dmg taken=217217)
INFO     |    3.35 |  72560 dmg by Chronoshift on Enemy(13, dmg taken=289777)
INFO     |    3.35 |  32021 dmg by Chronoshift on Enemy(

rime.chronoshift.cooldown = 137.0s  charges=0


## Damage report

In [6]:
print("Total damage per enemy:")
for enemy in enemies:
    print(f"  Enemy {enemy.id}: {enemy.damage_tracker.total:.0f}")

print("\nMain target — by source:")
for source, record in sorted(target.damage_tracker.by_source.items(), key=lambda x: -x[1].total):
    print(f"  {source}: {record.total:.0f}")

Total damage per enemy:
  Enemy 13: 1230189
  Enemy 14: 200735
  Enemy 15: 468914
  Enemy 16: 177082
  Enemy 17: 161233

Main target — by source:
  KindlingDoT: 1025900
  Chronoshift: 177116
  FlightOfTheNavir: 27173
